# **Eksperimen Machine Learning — Samuel Sitompul**

**Dataset**: Loan Approval Prediction  
**Sumber**: Kaggle — Loan Approval Prediction Dataset  
**Tujuan**: Membangun model klasifikasi untuk memprediksi apakah pengajuan pinjaman disetujui atau ditolak.

# **1. Perkenalan Dataset**

Dataset **Loan Approval Prediction** berisi data historis pengajuan pinjaman bank beserta hasilnya (Approved/Rejected). Dataset ini mencakup:

- **Jumlah data**: 4.269 baris
- **Jumlah fitur**: 12 kolom + 1 target
- **Sumber**: [Kaggle — Loan Approval Prediction](https://www.kaggle.com/datasets/architsharma01/loan-approval-prediction-dataset)

**Deskripsi kolom:**
| Kolom | Tipe | Deskripsi |
|---|---|---|
| loan_id | int | ID unik pinjaman (tidak digunakan untuk model) |
| no_of_dependents | int | Jumlah tanggungan pemohon (0–5) |
| education | str | Pendidikan (Graduate / Not Graduate) |
| self_employed | str | Status pekerjaan (Yes / No) |
| income_annum | int | Pendapatan tahunan (Rupee) |
| loan_amount | int | Jumlah pinjaman yang diminta |
| loan_term | int | Jangka waktu pinjaman (tahun) |
| cibil_score | int | Skor kredit CIBIL (300–900) |
| residential_assets_value | int | Nilai aset residensial |
| commercial_assets_value | int | Nilai aset komersial |
| luxury_assets_value | int | Nilai aset mewah |
| bank_asset_value | int | Nilai aset di bank |
| loan_status | str | **TARGET**: Approved / Rejected |

# **2. Import Library**

In [ ]:
# Library dasar
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualisasi
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Evaluasi
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)

print('Library berhasil di-import!')
print(f'Pandas version  : {pd.__version__}')
print(f'NumPy version   : {np.__version__}')

# **3. Memuat Dataset**

In [ ]:
# Load dataset
df = pd.read_csv('../loan_approval_dataset_raw.csv')

# Bersihkan whitespace di nama kolom dan nilai string
df.columns = df.columns.str.strip()
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

print(f'Shape dataset  : {df.shape}')
print(f'Kolom          : {df.columns.tolist()}')
df.head()

In [ ]:
# Informasi tipe data
df.info()

In [ ]:
# Statistik deskriptif
df.describe()

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, kita akan memahami karakteristik dataset melalui:
1. Distribusi variabel target
2. Distribusi fitur numerik
3. Hubungan antar fitur (korelasi)
4. Analisis fitur kategorikal

In [ ]:
# ── 4.1 Distribusi Target ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
loan_counts = df['loan_status'].value_counts()
axes[0].bar(loan_counts.index, loan_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribusi Status Pinjaman', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Jumlah')
for i, (k, v) in enumerate(loan_counts.items()):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(
    loan_counts.values, 
    labels=loan_counts.index, 
    autopct='%1.1f%%', 
    colors=['#2ecc71', '#e74c3c'],
    startangle=90
)
axes[1].set_title('Proporsi Status Pinjaman', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_target_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print(loan_counts)

In [ ]:
# ── 4.2 Distribusi Fitur Numerik ─────────────────────────────────────────────
num_cols = ['no_of_dependents', 'income_annum', 'loan_amount', 'loan_term',
            'cibil_score', 'residential_assets_value', 
            'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value']

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=30, color='#3498db', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_ylabel('Frekuensi')

plt.suptitle('Distribusi Fitur Numerik', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_numeric_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4.3 Boxplot: CIBIL Score vs Loan Status ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# CIBIL Score
approved = df[df['loan_status'] == 'Approved']['cibil_score']
rejected = df[df['loan_status'] == 'Rejected']['cibil_score']
axes[0].boxplot([approved, rejected], labels=['Approved', 'Rejected'], 
                patch_artist=True,
                boxprops=dict(facecolor='#3498db', alpha=0.7))
axes[0].set_title('CIBIL Score vs Loan Status', fontsize=13, fontweight='bold')
axes[0].set_ylabel('CIBIL Score')

# Income vs Status
approved_inc = df[df['loan_status'] == 'Approved']['income_annum']
rejected_inc = df[df['loan_status'] == 'Rejected']['income_annum']
axes[1].boxplot([approved_inc, rejected_inc], labels=['Approved', 'Rejected'],
                patch_artist=True,
                boxprops=dict(facecolor='#2ecc71', alpha=0.7))
axes[1].set_title('Income Annum vs Loan Status', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Income (Rupee)')

plt.tight_layout()
plt.savefig('eda_boxplot.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4.4 Korelasi antar Fitur Numerik ─────────────────────────────────────────
df_temp = df[num_cols].copy()

plt.figure(figsize=(10, 8))
corr_matrix = df_temp.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    mask=mask,
    center=0,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
plt.title('Heatmap Korelasi Fitur Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4.5 Fitur Kategorikal ────────────────────────────────────────────────────
cat_cols = ['education', 'self_employed']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors_map = {'Approved': '#2ecc71', 'Rejected': '#e74c3c'}

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df['loan_status'])
    ct.plot(kind='bar', ax=axes[i], color=[colors_map['Approved'], colors_map['Rejected']], 
            edgecolor='white', alpha=0.8)
    axes[i].set_title(f'{col} vs Loan Status', fontsize=13, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Jumlah')
    axes[i].tick_params(axis='x', rotation=0)
    axes[i].legend(['Approved', 'Rejected'])

plt.tight_layout()
plt.savefig('eda_categorical.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4.6 Missing Values & Duplikat ────────────────────────────────────────────
print('=== Cek Missing Values ===')
print(df.isnull().sum())

print(f'\n=== Cek Duplikat ===')
print(f'Jumlah duplikat: {df.duplicated().sum()}')

print('\nKesimpulan: Dataset sangat bersih, tidak ada missing values maupun duplikat.')

# **5. Data Preprocessing**

Tahapan preprocessing yang dilakukan:
1. **Menghapus kolom tidak relevan** (`loan_id`)
2. **Encoding kategorikal** — LabelEncoder untuk `education`, `self_employed`, `loan_status`
3. **Standarisasi fitur numerik** — StandardScaler
4. **Train-Test Split** — 80% train, 20% test
5. **Simpan hasil** ke CSV

In [ ]:
# ── 5.1 Drop kolom tidak relevan ─────────────────────────────────────────────
df_clean = df.copy()
df_clean = df_clean.drop(columns=['loan_id'])
print(f'Shape setelah drop loan_id: {df_clean.shape}')
df_clean.head(3)

In [ ]:
# ── 5.2 Label Encoding ────────────────────────────────────────────────────────
label_encoders = {}
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
print(f'Kolom kategorikal: {cat_cols}')

for col in cat_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    label_encoders[col] = le
    print(f'  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

df_clean.head(3)

In [ ]:
# ── 5.3 Standarisasi Fitur Numerik ────────────────────────────────────────────
TARGET_COL = 'loan_status'
num_cols_to_scale = df_clean.select_dtypes(include=[np.number]).columns.tolist()
num_cols_to_scale = [c for c in num_cols_to_scale if c != TARGET_COL]
print(f'Kolom yang di-scale: {num_cols_to_scale}')

scaler = StandardScaler()
df_clean[num_cols_to_scale] = scaler.fit_transform(df_clean[num_cols_to_scale])

print('\nStatistik setelah scaling:')
df_clean[num_cols_to_scale].describe().round(3)

In [ ]:
# ── 5.4 Train-Test Split ──────────────────────────────────────────────────────
X = df_clean.drop(columns=[TARGET_COL])
y = df_clean[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'y_train distribusi: {y_train.value_counts().to_dict()}')
print(f'y_test distribusi : {y_test.value_counts().to_dict()}')

In [ ]:
# ── 5.5 Simpan Hasil Preprocessing ───────────────────────────────────────────
output_path = 'loan_approval_dataset_preprocessing.csv'
df_clean.to_csv(output_path, index=False)
print(f'Dataset preprocessing berhasil disimpan ke: {output_path}')
print(f'Shape: {df_clean.shape}')
df_clean.head()

# **Kesimpulan Eksperimen**

Dataset **Loan Approval Prediction** telah berhasil diproses melalui tahapan:

| Tahapan | Detail |
|---|---|
| Data Loading | CSV berhasil dimuat, 4269 baris × 13 kolom |
| EDA | Distribusi target seimbang (62%/38%), CIBIL Score adalah fitur paling diskriminatif |
| Drop kolom | `loan_id` dihapus |
| Label Encoding | `education`, `self_employed`, `loan_status` |
| Standarisasi | StandardScaler pada 9 fitur numerik |
| Split | 80% train (3415), 20% test (854) |
| Output | `loan_approval_dataset_preprocessing.csv` (4269 × 12) |

Dataset siap digunakan untuk pelatihan model machine learning.